# 原始 Stage1 39 特征 + AMD/NVIDIA 原始前瞻收入增长广度 H5 信息门

单一来源坐标、完整日历、训练期冗余预检，2023–2026 为复用开发历史。

In [1]:
import os,json,hashlib,sys,sklearn
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd();card=root/'research/stage1_amd_nvda_guidance_breadth_h5_information_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and sys.version_info[:2]==(3,12) and sklearn.__version__=='1.9.1'
parent=plan['parent'];parent_card=root/parent['source_card']
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:assert sha(parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/name)==parent[key]
assert json.loads((root/parent['nas_readback']).read_text())['NAS_archive_sha256']==parent['nas_archive_sha256']
spec=plan['source'];assert sha(root/spec['summary'])==spec['summary_sha256'];source_summary=json.loads((root/spec['summary']).read_text());assert source_summary['recorder_id']==spec['recorder_id'] and source_summary['nas_readback']['verified']
source_artifact=Path(source_summary['recorder_uri'])/source_summary['experiment_id']/source_summary['recorder_id']/'artifacts'/spec['feature_artifact'];assert sha(source_artifact)==spec['mapped_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(parent_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
with R.uri_context(source_summary['recorder_uri']):mapped=R.get_recorder(recorder_id=source_summary['recorder_id'],experiment_id=source_summary['experiment_id']).load_object(spec['feature_artifact'])
assert len(mapped)==3966 and mapped.index.is_unique and int(mapped.loc['2011-01-03':'2022-12-30','source_known'].sum())==2881 and int(mapped.loc['2023-01-03':'2026-10-07','source_known'].sum())==944
feature=mapped['AMD_NVDA_forward_growth_positive_breadth'].astype(float).reindex(frame.index).rename(plan['single_mechanism']['feature'])
assert feature.notna().equals(mapped.source_known.reindex(frame.index).fillna(False))
forward=frame.soxx_forward_open_return.astype(float);parts=[forward.shift(-i) for i in range(5)];valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts]);label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True);cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna();held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert train_dates.sum()==plan['cohorts']['expected_mature_train_rows'] and held_dates.sum()==plan['cohorts']['expected_mature_held_rows']
novel_train=train_dates & feature.notna();assert novel_train.sum()>2500
corrs={}
for name in base_names:
 x=base[name].astype(float).loc[novel_train].to_numpy(float);y=feature.loc[novel_train].to_numpy(float);mask=np.isfinite(x)&np.isfinite(y)
 if mask.sum()>100 and np.std(x[mask])>0 and np.std(y[mask])>0:corrs[name]=float(np.corrcoef(x[mask],y[mask])[0,1])
assert len(corrs)>30;max_abs_training_pairwise_Pearson=max(abs(x) for x in corrs.values());assert max_abs_training_pairwise_Pearson<plan['training_novelty_gate']['maximum_absolute_pairwise_Pearson_to_each_original39']
augmented=base[base_names].copy();augmented[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates,'source_known':mapped.source_known.reindex(frame.index).fillna(False)},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1);static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])});train_prepared=dataset.prepare('train',col_set=['feature','label','meta']);held_prepared=dataset.prepare('held',col_set=['feature','label','meta']);assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'Stage1_AMD_NVDA_guidance_breadth_DatasetH_label_maturity_preflight_passed_conditional_source_no_fit','plan_sha256':sha(card/'plan.json'),'parent_recorder_id':parent['recorder_id'],'source_recorder_id':source_summary['recorder_id'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_unknown_rows':int(feature.loc[train_dates].isna().sum()),'candidate_held_unknown_rows':int(feature.loc[held_dates].isna().sum()),'max_abs_training_pairwise_Pearson':max_abs_training_pairwise_Pearson,'training_pairwise_valid_original_features':len(corrs),'source_Recorder_object_loads':5,'DatasetH_builds':1,'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False,'historical_first_public_qualified':False}
(card/'preflight_result.json').write_text(json.dumps(report,indent=2)+'\n');print({k:report[k] for k in ('status','training_mature_rows','held_mature_rows','candidate_train_unknown_rows','candidate_held_unknown_rows')})


2026/10/23 06:29:21 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[5663:MainThread](2026-10-23 06:29:21,643) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[5663:MainThread](2026-10-23 06:29:21,645) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[5663:MainThread](2026-10-23 06:29:21,645) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[5663:MainThread](2026-10-23 06:29:22,914) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[5663:MainThread](2026-10-23 06:29:22,914) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[5663:MainThread](2026-10-23 06:29:22,915) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'status': 'Stage1_AMD_NVDA_guidance_breadth_DatasetH_label_maturity_preflight_passed_conditional_source_no_fit', 'training_mature_rows': 3014, 'held_mature_rows': 940, 'candidate_train_unknown_rows': 139, 'candidate_held_unknown_rows': 0}


## 冻结拟合与成对信息检验

只有独立设计审查与预检哈希匹配后执行。

In [2]:
import shlex,tarfile,tempfile,subprocess
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings
review_path=card/'design_review_final.json';assert review_path.exists()
review=json.loads(review_path.read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());preflight_source=''.join(notebook['cells'][1]['source']);preflight_hash=hashlib.sha256(preflight_source.encode()).hexdigest();fit_source=''.join(notebook['cells'][3]['source']);fit_hash=hashlib.sha256(fit_source.encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==sha(card/'plan.json') and review['preflight_sha256']==sha(card/'preflight_result.json') and review['preflight_cell_source_sha256']==preflight_hash and review['fit_cell_source_sha256']==fit_hash
assert report==json.loads((card/'preflight_result.json').read_text()) and not report['historical_first_public_qualified']
state_path=card/'run_state.json';tracking=card/'qlib_recorder';assert not state_path.exists() and not tracking.exists()
state={'plan_sha256':sha(card/'plan.json'),'fit_cell_source_sha256':fit_hash,'fit_attempts':[],'fits_done':[],'prediction_attempts':[],'predictions_done':[],'bootstrap_attempted':False,'new_native_accounts':0}
state_path.write_text(json.dumps(state,indent=2)+'\n')
tr=train_prepared.loc[train_prepared[('meta','train_eligible')].astype(bool)];he=held_prepared.loc[held_prepared[('meta','held_eligible')].astype(bool)]
assert len(tr)==report['training_mature_rows']==3014 and len(he)==report['held_mature_rows']==940
yname='soxx_forward_five_open_log_return';feature_name=plan['single_mechanism']['feature'];ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
inputs={'baseline':(tr['feature'][base_names].to_numpy(float),he['feature'][base_names].to_numpy(float)),'candidate':(tr['feature'][base_names+[feature_name]].to_numpy(float),he['feature'][base_names+[feature_name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][0][:,-1]).sum()==len(inputs['candidate'][0])-report['candidate_train_unknown_rows'] and np.isfinite(inputs['candidate'][1][:,-1]).sum()==len(inputs['candidate'][1])-report['candidate_held_unknown_rows']
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));state_path.write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True)
 recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,derived_coordinate=feature,source_hashes={'AMD_NVDA_original_guidance_breadth':spec['summary_sha256'],'original39':parent['feature_artifact_sha256']})
 assert len(recorder.load_object('inputs/dataset').prepare('train',col_set=['feature','label','meta']))==report['train_segment_rows']
 for family in ('baseline','candidate'):
  assert family not in state['fit_attempts'] and len(state['fit_attempts'])<plan['budget']['model_fit_attempts_max']
  state['fit_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));model.fit(inputs[family][0],ytr)
  recorder.save_objects(artifact_path='models',**{family:model});models[family]=model;state['fits_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  state['prediction_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  prediction=model.predict(inputs[family][1]);assert np.isfinite(prediction).all()
  recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})});predictions[family]=prediction
  state['predictions_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n');recorder.log_metrics(completed_fits=len(state['fits_done']),completed_predictions=len(state['predictions_done']))
 assert len(state['fits_done'])==len(state['predictions_done'])==2
 dates=pd.DatetimeIndex(he.index.get_level_values('datetime'));daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
 daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2;daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2;daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
 assert daily.signal_date.is_monotonic_increasing and len(daily)==940
 halves=[daily.loc[daily.signal_date.lt(pd.Timestamp('2025-01-01'))],daily.loc[daily.signal_date.ge(pd.Timestamp('2025-01-01'))]];assert min(map(len,halves))>300
 held_calendar=pd.DatetimeIndex(held_prepared.index.get_level_values('datetime'));assert len(held_calendar)==report['held_segment_rows']==944 and held_calendar.is_unique and held_calendar.is_monotonic_increasing
 full_calendar=pd.DataFrame({'signal_date':held_calendar}).merge(daily[['signal_date','paired_mse_improvement']],on='signal_date',how='left',validate='one_to_one')
 assert len(full_calendar)==944 and full_calendar.paired_mse_improvement.notna().sum()==940 and full_calendar.paired_mse_improvement.isna().sum()==4
 assert full_calendar.paired_mse_improvement.iloc[:-4].notna().all() and full_calendar.paired_mse_improvement.iloc[-4:].isna().all()
 bootstrap=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempted']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
 interval,_,replicates=full_calendar_bootstrap(full_calendar.paired_mse_improvement.to_numpy(float),mean_block=bootstrap['mean_block_sessions'],resamples=bootstrap['draws'],seed=bootstrap['seed'],alpha=bootstrap['one_sided_alpha'])
 assert interval['calendar_rows']==944 and interval['scored_rows']==940
 complete_year_means={str(y):float(daily.loc[daily.signal_date.dt.year.eq(y),'paired_mse_improvement'].mean()) for y in (2024,2025)};assert all(daily.signal_date.dt.year.eq(y).sum()>=240 for y in (2024,2025))
 coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
 metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),'candidate_held_mse':float(daily.candidate_squared_error.mean()),'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],'complete_2024_2025_paired_means':complete_year_means,'candidate_standardized_coordinate_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
 gates={'coefficient_positive':coefficient>0,'full_held_mean_positive':metrics['held_paired_mse_improvement']>0,'both_halves_positive':all(x>0 for x in metrics['held_half_paired_improvements']),'each_complete_2024_2025_year_mean_positive':all(x>0 for x in complete_year_means.values()),'bootstrap_lower_positive':metrics['bootstrap_lower']>0};passed=bool(all(gates.values()))
 daily['quarter']=daily.signal_date.dt.to_period('Q').astype(str);quarters=[{'quarter':q,'sessions':len(g),'paired_mean_mse_improvement':float(g.paired_mse_improvement.mean())} for q,g in daily.groupby('quarter',sort=True)]
 recorder.save_objects(artifact_path='analysis/daily',held=daily,full_calendar=full_calendar);recorder.save_objects(artifact_path='analysis',metrics=metrics,gates=gates,quarters=quarters,bootstrap_replicates=replicates,information_gate_pass=passed)
 recorder.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':'conditional_development_AMD_NVDA_guidance_breadth_information_pass_first_public_pending_no_account' if passed else 'not_promoted_fixed_AMD_NVDA_guidance_breadth_H5_information_failed','objective_achieved':False,'information_gate_pass':passed,'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'gates':gates,'quarterly_information_attribution':quarters,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'historical_first_public_qualified':False,'reused_market_history':True,'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'parent_feature_sha256':parent['feature_artifact_sha256'],'source_summary_sha256':spec['summary_sha256'],'artifact_policy':plan['artifact_policy'],'limitations':['AMD/NVDA original SEC filing-next-signal map lacks exact issuer first-public and SEC public-visibility clocks','Information MSE is not fee-net SOXX funded account result','Reused held 2023-2026 is development feedback, not independent forward']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'result_review.json').write_text(json.dumps({'status':summary['decision'],'plan_sha256':summary['plan_sha256'],'recorder_id':state['recorder_id'],'metrics':metrics,'gates':gates,'new_model_fits':2,'new_native_accounts':0},indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='amd-nvda-guidance-info-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/models/baseline') for name in names) and any(name.endswith('/artifacts/models/candidate') for name in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['completed']=True;state['nas_archive_sha256']=evidence['sha256'];state_path.write_text(json.dumps(state,indent=2)+'\n')
print({k:summary[k] for k in ('decision','information_gate_pass','new_model_fits','new_native_accounts','recorder_id')});print(metrics)


[5663:MainThread](2026-10-23 06:29:22,932) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_amd_nvda_guidance_breadth_h5_information_20261023.


[5663:MainThread](2026-10-23 06:29:22,933) INFO - qlib.workflow - [exp.py:258] - Experiment 138902687451024729 starts running ...


[5663:MainThread](2026-10-23 06:29:23,006) INFO - qlib.workflow - [recorder.py:345] - Recorder 7016da6c1f6f47b594af31847bbbc3c3 starts running under Experiment 138902687451024729 ...


[5663:MainThread](2026-10-23 06:29:25,959) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'decision': 'not_promoted_fixed_AMD_NVDA_guidance_breadth_H5_information_failed', 'information_gate_pass': False, 'new_model_fits': 2, 'new_native_accounts': 0, 'recorder_id': '7016da6c1f6f47b594af31847bbbc3c3'}
{'baseline_held_mse': 0.0025262669365982205, 'candidate_held_mse': 0.00253848390478999, 'held_paired_mse_improvement': -1.2216968191769588e-05, 'held_half_paired_improvements': [7.2458256311711765e-06, -3.452364056418115e-05], 'complete_2024_2025_paired_means': {'2024': 7.35074119724125e-06, '2025': -2.052319410953872e-06}, 'candidate_standardized_coordinate_coefficient': 0.001593911619638669, 'bootstrap_lower': -6.504925902926443e-05, 'bootstrap_upper': 2.5027387408072913e-05, 'bootstrap': {'paired_mean_loss_improvement': -1.2216968191769587e-05, 'lower_bound': -6.504925902926443e-05, 'upper_bound': 2.5027387408072913e-05, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 944, 'scored_rows': 940}}


## 中文结论

# 原 Stage1 39 项 + AMD/NVIDIA 原始前瞻收入增长广度：H5 信息检验

冻结完整 Stage1 原始 39 项输入为对照，只增加 AMD/NVIDIA 两家公司原始 SEC 季度前瞻收入增长的正值广度一个坐标。Qlib 训练/复用评价日历分别 3020/944 日，成熟标签 3014/940 日；训练期来源未知 139 日保留并由训练期中位数插补，评价期来源全覆盖。新增坐标对原 39 项训练期最大逐项相关绝对值 0.1286。

固定两次 Ridge 拟合、一次完整 944 日日历的 63 日区块 bootstrap 后，对照 MSE 0.002526267，候选 0.002538484，成对改善 -1.22169682e-05。新增标准化系数 +0.00159391 符合假设方向，但整体、后半段、完整 2025 年和校正区块下界均未通过；两半分别 +7.24583e-06/-3.45236e-05，完整 2024/2025 年分别 +7.35074e-06/-2.05232e-06，校正下界 -6.50493e-05。五项固定门槛只通过系数方向，决定 `not_promoted_fixed_AMD_NVDA_guidance_breadth_H5_information_failed`，0 新资金账户。

区间对比表明来源状态本身没有区分失败：2025Q4 和 2026Q3 的广度均为 1，前者信息改善 +2.15399e-05，后者恶化 -7.96998e-05；2026Q2 广度均值 0.806，信息恶化 -1.61922e-04。这些是复用历史的失败归因，不能用季度或状态选择开关。下一步回到完整 Stage1 的现金机会与 SOXL 相对选择两项损失，对照差/好区间、行动时信息及 39 项既有输入，冻结不同可证伪机制；不调整本版符号、四个月有效期、发行人组合、H5 或窗口救援。

Qlib Recorder `7016da6c1f6f47b594af31847bbbc3c3` 的模型、指标和 944/940 日历读回一致；独立复核确认失败。Yanxuan NAS 归档 SHA-256 `3f0916910a8ed36a45c92a590156d2812c92474125431c5d270e1bab2bf4784e` 已远端逐项恢复核验。SEC/发行人精确首次公开时钟仍未合格；2023–2026 是复用开发历史，未产生完整 Stage1 衍生 0bp Tiered 账户、2/5bp 或独立前向证据。SOXX v2 目标未达成，整体 Goal 继续。
